# 💳 Financial Transactions — Data Extraction Pipeline
**Objective:** Extract and structure financial transaction data from Kaggle to generate analysis-ready CSVs for Power BI dashboards.

**Focus areas:**
- Customer segmentation (spending behavior + demographics)
- Merchant & transaction trend analysis (patterns over time)

In [1]:
import kagglehub
import pandas as pd
# Download latest version
path = kagglehub.dataset_download("computingvictor/transactions-fraud-datasets")

print("Path to dataset files:", path)

/usr/local/python/3.12.1/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Path to dataset files: /home/codespace/.cache/kagglehub/datasets/computingvictor/transactions-fraud-datasets/versions/1


## 📥 Step 1: Load the Data
Downloading the dataset via `kagglehub` and loading each file into a Pandas DataFrame.
We select only the relevant columns from `transactions_data.csv` to reduce memory usage.

In [2]:
transactions_df = pd.read_csv(
    "/home/codespace/.cache/kagglehub/datasets/computingvictor/transactions-fraud-datasets/versions"
    "/1/transactions_data.csv",usecols=["id","date","client_id","card_id","amount","merchant_id","mcc","errors"])

In [3]:
users_df=pd.read_csv("/home/codespace/.cache/kagglehub/datasets/computingvictor/transactions-fraud-datasets/versions/1/users_data.csv")
users_df.head(5)

,id,current_age,retirement_age,birth_year,birth_month,gender,address,latitude,longitude,per_capita_income,yearly_income,total_debt,credit_score,num_credit_cards
0,825,53,66,1966,11,Female,462 Rose Lane,34.15,-117.76,$29278,$59696,$127613,787,5
1,1746,53,68,1966,12,Female,3606 Federal Boulevard,40.76,-73.74,$37891,$77254,$191349,701,5
2,1718,81,67,1938,11,Female,766 Third Drive,34.02,-117.89,$22681,$33483,$196,698,5
3,708,63,63,1957,1,Female,3 Madison Street,40.71,-73.99,$163145,$249925,$202328,722,4
4,1164,43,70,1976,9,Male,9620 Valley Stream Drive,37.76,-122.44,$53797,$109687,$183855,675,1


In [4]:
cards_df=pd.read_csv("/home/codespace/.cache/kagglehub/datasets/computingvictor/transactions-fraud-datasets/versions/1/cards_data.csv")
cards_df.head(5)

,id,client_id,card_brand,card_type,card_number,expires,cvv,has_chip,num_cards_issued,credit_limit,acct_open_date,year_pin_last_changed,card_on_dark_web
0,4524,825,Visa,Debit,4344676511950444,12/2022,623,YES,2,$24295,09/2002,2008,No
1,2731,825,Visa,Debit,4956965974959986,12/2020,393,YES,2,$21968,04/2014,2014,No
2,3701,825,Visa,Debit,4582313478255491,02/2024,719,YES,2,$46414,07/2003,2004,No
3,42,825,Visa,Credit,4879494103069057,08/2024,693,NO,1,$12400,01/2003,2012,No
4,4659,825,Mastercard,Debit (Prepaid),5722874738736011,03/2009,75,YES,1,$28,09/2008,2009,No


In [5]:
import pandas as pd
import matplotlib.pyplot as plt
import json

# 1. Correctly open and load the JSON file's data
file_path = "/home/codespace/.cache/kagglehub/datasets/computingvictor/transactions-fraud-datasets/versions/1/mcc_codes.json"
with open(file_path, 'r') as f:
    mcc_data = json.load(f)

# 2. Convert key-value pairs to rows and name your columns
mcc_df = pd.DataFrame(list(mcc_data.items()), columns=['mcc_code', 'description'])

# 3. Set your newly named business description or code as the index
mcc_df.set_index('mcc_code', inplace=True)

# Print clean structural output
print(mcc_df.head())

                                   description
mcc_code                                      
5812             Eating Places and Restaurants
5541                          Service Stations
7996      Amusement Parks, Carnivals, Circuses
5411              Grocery Stores, Supermarkets
4784                     Tolls and Bridge Fees


## 🧹 Step 2: Clean the Data
Inspecting data types, fixing the `amount` column (stored as string with `$` prefix), and handling null values.

In [6]:
transactions_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 13305915 entries, 0 to 13305914
Data columns (total 8 columns):
 #   Column       Dtype
---  ------       -----
 0   id           int64
 1   date         str  
 2   client_id    int64
 3   card_id      int64
 4   amount       str  
 5   merchant_id  int64
 6   mcc          int64
 7   errors       str  
dtypes: int64(5), str(3)
memory usage: 812.1 MB


In [7]:
users_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 14 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   id                 2000 non-null   int64  
 1   current_age        2000 non-null   int64  
 2   retirement_age     2000 non-null   int64  
 3   birth_year         2000 non-null   int64  
 4   birth_month        2000 non-null   int64  
 5   gender             2000 non-null   str    
 6   address            2000 non-null   str    
 7   latitude           2000 non-null   float64
 8   longitude          2000 non-null   float64
 9   per_capita_income  2000 non-null   str    
 10  yearly_income      2000 non-null   str    
 11  total_debt         2000 non-null   str    
 12  credit_score       2000 non-null   int64  
 13  num_credit_cards   2000 non-null   int64  
dtypes: float64(2), int64(7), str(5)
memory usage: 218.9 KB


### Fix `amount` Data Type
The `amount` column is loaded as a string. Stripping the `$` sign and casting to `float64`.

In [8]:
transactions_df["amount"] = transactions_df["amount"].str.replace("$", "", regex=False)

In [9]:
transactions_df=transactions_df.astype({
    "amount": "float64"})

In [10]:
transactions_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 13305915 entries, 0 to 13305914
Data columns (total 8 columns):
 #   Column       Dtype  
---  ------       -----  
 0   id           int64  
 1   date         str    
 2   client_id    int64  
 3   card_id      int64  
 4   amount       float64
 5   merchant_id  int64  
 6   mcc          int64  
 7   errors       str    
dtypes: float64(1), int64(5), str(2)
memory usage: 812.1 MB


### Null Value Inspection
Checking for missing values across all four tables before deciding how to handle them.

In [11]:
transactions_df.isnull().sum()

id                    0
date                  0
client_id             0
card_id               0
amount                0
merchant_id           0
mcc                   0
errors         13094522
dtype: int64

In [12]:
transactions_df["errors"].unique()

<StringArray>
[                                                  nan,
                                    'Technical Glitch',
                                      'Bad Expiration',
                                     'Bad Card Number',
                                'Insufficient Balance',
                                             'Bad PIN',
                                             'Bad CVV',
                                         'Bad Zipcode',
               'Insufficient Balance,Technical Glitch',
                        'Bad PIN,Insufficient Balance',
                            'Bad PIN,Technical Glitch',
                     'Bad Expiration,Technical Glitch',
                      'Bad Card Number,Bad Expiration',
                'Bad Card Number,Insufficient Balance',
                 'Bad Expiration,Insufficient Balance',
                             'Bad Card Number,Bad CVV',
                            'Bad CVV,Technical Glitch',
                              'Bad

### Handle Nulls in `errors`
Null entries in `errors` indicate clean transactions with no issue.
Filling with `"blank"` to keep the column meaningful in Power BI filters.

In [13]:
transactions_df["errors"]=transactions_df["errors"].fillna("blank")

In [14]:
transactions_df.isnull().sum()

id             0
date           0
client_id      0
card_id        0
amount         0
merchant_id    0
mcc            0
errors         0
dtype: int64

In [15]:
cards_df.isnull().sum()

id                       0
client_id                0
card_brand               0
card_type                0
card_number              0
expires                  0
cvv                      0
has_chip                 0
num_cards_issued         0
credit_limit             0
acct_open_date           0
year_pin_last_changed    0
card_on_dark_web         0
dtype: int64

In [16]:
users_df.isnull().sum()

id                   0
current_age          0
retirement_age       0
birth_year           0
birth_month          0
gender               0
address              0
latitude             0
longitude            0
per_capita_income    0
yearly_income        0
total_debt           0
credit_score         0
num_credit_cards     0
dtype: int64

In [17]:
mcc_df.isnull().sum()

description    0
dtype: int64

## 🗄️ Step 3: Load into SQLite
Writing all four tables into a local SQLite database (`bank_data.db`).

**Why SQLite?**
- Enables clean multi-table SQL joins without chained pandas merges
- Allows re-running queries without reloading raw CSVs
- Mirrors the structure of a real data warehouse

Large files are written in **chunks of 50,000 rows** to avoid memory overflows.

In [18]:
import json
import sqlite3
import pandas as pd

# 1. Connect to the database file
conn = sqlite3.connect('bank_data.db')

# --- PART A: PROCESS MCC CODES (Small JSON file - safe to read all at once) ---
json_path = "/home/codespace/.cache/kagglehub/datasets/computingvictor/transactions-fraud-datasets/versions/1/mcc_codes.json"
with open(json_path, 'r') as f:
    mcc_data = json.load(f)

mcc_df = pd.DataFrame(list(mcc_data.items()), columns=['mcc_code', 'description'])
mcc_df.to_sql('mcc', conn, if_exists='replace', index=False)
print("✔ MCC codes table written.")


# --- PART B: PROCESS CARDS DATA (Medium CSV file - read in chunks) ---
cards_path = "/home/codespace/.cache/kagglehub/datasets/computingvictor/transactions-fraud-datasets/versions/1/cards_data.csv"
chunk_size = 50000

# Write the first chunk with 'replace' to clear out old data, then append the rest
first_chunk = True
for chunk in pd.read_csv(cards_path, chunksize=chunk_size):
    if first_chunk:
        chunk.to_sql('cards', conn, if_exists='replace', index=False)
        first_chunk = False
    else:
        chunk.to_sql('cards', conn, if_exists='append', index=False)
print("✔ Cards table written in chunks.")

total_rows = len(transactions_df)
chunk_size = 50000
table_name = "transactions"

for i in range(0, total_rows, chunk_size):
    # Extract a temporary slice of rows
    df_slice = transactions_df.iloc[i : i + chunk_size]
    
    if i == 0:
        # Create a fresh table with 'replace' on the first batch
        df_slice.to_sql(table_name, conn, if_exists='replace', index=False)
    else:
        # Append all subsequent batches to the bottom of the table
        df_slice.to_sql(table_name, conn, if_exists='append', index=False)

print(f"✔ Successfully uploaded all {total_rows} rows to the database!")

csv_path = "/home/codespace/.cache/kagglehub/datasets/computingvictor/transactions-fraud-datasets/versions/1/users_data.csv"
first_chunk = True
for chunk in pd.read_csv(csv_path, chunksize=chunk_size):
    if first_chunk:
        chunk.to_sql('users', conn, if_exists='replace', index=False)
        first_chunk = False
    else:
        chunk.to_sql('users', conn, if_exists='append', index=False)
print("✔ Users table written in chunks.")


# --- PART D: CLEAN UP ---
print(" All tables successfully written to fraud_data.db without crashing!")
conn.close()


✔ MCC codes table written.
✔ Cards table written in chunks.
✔ Successfully uploaded all 13305915 rows to the database!
✔ Users table written in chunks.
 All tables successfully written to fraud_data.db without crashing!


## 👤 Step 4: Customer Segmentation Query
Joining transactions with user demographics to build a **per-customer spending profile**.

Each row = one customer, aggregated across all their transactions, enriched with:
- Spending stats (total, average, largest transaction)
- Demographic info (age, gender, income, debt, number of cards)
- Merchant category preference

**Output:** `customer analysis.csv` → ready for Power BI segmentation visuals.

In [19]:
conn = sqlite3.connect('bank_data.db')

sql_query = """SELECT
    t.client_id,
    COUNT(*) AS total_transactions,
    SUM(t.amount) AS total_spent,
    AVG(t.amount) AS avg_transaction,
    MAX(t.amount) AS largest_transaction,
    u.current_age,
    u.gender,
    u.yearly_income,
    u.total_debt,
    u.num_credit_cards,m.description
FROM transactions t
INNER JOIN users u
    ON t.client_id = u.id
INNER JOIN mcc m
    ON t.mcc = mcc_code
GROUP BY
    t.client_id,
    u.current_age,
    u.gender,
    u.yearly_income,
    u.total_debt,
    u.num_credit_cards"""
customer_spending_df = pd.read_sql_query(sql_query, conn)

conn.close()

In [20]:
customer_spending_df.head(5)

,client_id,total_transactions,total_spent,avg_transaction,largest_transaction,current_age,gender,yearly_income,total_debt,num_credit_cards,description
0,0,12795,625799.67,48.909705,1128.47,33,Male,$59613,$36199,4,Miscellaneous Fabricated Metal Products
1,1,10073,336187.37,33.375099,937.15,43,Female,$45360,$14587,3,"Bolt, Nut, Screw, Rivet Manufacturing"
2,2,10612,291534.27,27.472132,519.02,48,Male,$27447,$80850,5,Steelworks
3,3,6001,280685.46,46.773114,990.20,49,Male,$27943,$18693,4,Pottery and Ceramics
4,4,15043,595722.36,39.601300,1624.15,54,Female,$76431,$115362,5,Hospitals


In [21]:
customer_spending_df.to_csv('customer analysis.csv', index=False)

## 📈 Step 5: Merchant & Transaction Trends Query
Aggregating transactions by **month and merchant category** to surface patterns over time.

Each row = one month + merchant category combination, with:
- Transaction volume
- Total and average spend

**Output:** `merchant_df` → ready for time-series and category breakdown visuals in Power BI.

In [22]:
conn = sqlite3.connect('bank_data.db')

sql_query = """
SELECT
    STRFTIME('%Y-%m', t.date) AS month,
    t.mcc AS merchant_category,

    COUNT(*) AS transactions,
    SUM(t.amount) AS total_spent,
    AVG(t.amount) AS avg_spent

FROM transactions t
GROUP BY
    month,
    merchant_category
ORDER BY
    month;
"""
merchant_df = pd.read_sql_query(sql_query, conn)

conn.close()

In [23]:
merchant_df.head()

,month,merchant_category,transactions,total_spent,avg_spent
0,2010-01,1711,30,3976.71,132.557000
1,2010-01,3000,21,13215.43,629.306190
2,2010-01,3001,24,18043.09,751.795417
3,2010-01,3005,4,2582.63,645.657500
4,2010-01,3006,2,1460.65,730.325000


In [25]:
merchant_df.to_csv('merchant & transaction.csv')